In [1]:
from unsloth import FastLanguageModel
import torch

max_seq_length = 16384  # Choose any! We auto support RoPE Scaling internally!
dtype = None  # None for auto detection. Float16 for Tesla T4, V100, Bfloat16 for Ampere+
load_in_4bit = True  # Use 4bit quantization to reduce memory usage. Can be False.
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name="unsloth/Phi-4-unsloth-bnb-4bit",
    max_seq_length=max_seq_length,
    dtype=dtype,
    load_in_4bit=load_in_4bit, device_map="cuda"
)
FastLanguageModel.for_inference(model)  # Enable native 2x faster inference
print("Load Finished")


🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.


C:\Users\HatsuneMiku\anaconda3\envs\spmgpt\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
W0220 17:00:43.418000 5980 site-packages\torch\distributed\elastic\multiprocessing\redirects.py:29] NOTE: Redirects are currently not supported in Windows or MacOs.


🦥 Unsloth Zoo will now patch everything to make training faster!
==((====))==  Unsloth 2026.2.1: Fast Llama patching. Transformers: 4.57.6.
   \\   /|    NVIDIA GeForce RTX 5090. Num GPUs = 1. Max memory: 31.842 GB. Platform: Windows.
O^O/ \_/ \    Torch: 2.7.0+cu128. CUDA: 12.0. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = TRUE. FA [Xformers = 0.0.30. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Loading checkpoint shards: 100%|██████████| 3/3 [00:06<00:00,  2.15s/it]


Load Finished


In [2]:
import json
from tqdm.autonotebook import tqdm

In [3]:
answer_system_prompt = (
    "Here is a reference text for you. Read it carefully, and then answer my question in a professional academic manner. "
    "Your answer should be detailed, well-structured, and easy to understand. Please analyze the content to offer deeper interpretation and elaboration based on the text."
    "Here is the information text: {}")


def generate_answer(question_text, information_text, max_length=1500):
    user_msg = {"role": "user", "content": question_text}

    prompt_text = [
        {"role": "system", "content": answer_system_prompt.format(information_text)},
    ]
    prompt_text.append(user_msg)

    inputs = tokenizer.apply_chat_template(
        prompt_text,
        tokenize=True,
        add_generation_prompt=True,  # Must add for generation
        return_tensors="pt",
    ).to("cuda")

    print(f"input token count is {len(inputs[0])}")

    inputs_text = tokenizer.decode(inputs[0])
    # print(inputs_text)
    outputs = model.generate(input_ids=inputs, max_new_tokens=max_length, use_cache=True,
                             temperature=0.9, min_p=0.1)

    outputs = tokenizer.decode(outputs[0]).split(inputs_text)[-1]
    return outputs.split(tokenizer.eos_token)[0]




In [4]:
def __write_answer_to_journal_dataset(path):
    with open(path, "r") as json_file:
        data = json.load(json_file)

    answer_list = []
    if data["type"] == "journal" and data["answered"] == False:
        for it in tqdm(data["questions"]):
            print("question:", it)
            answer = generate_answer(question_text=it, information_text=data["text"])
            print("answer:", answer)
            answer_list.append(answer)

        data["answers"] = answer_list
        data["answered"] = True
        data["distilled"] = False
        with open(path, "w") as json_file:
            json.dump(data, json_file, indent=4)


def __write_answer_to_book_dataset(path):
    with open(path, "r") as json_file:
        data = json.load(json_file)

    if data["type"] == "book" and data["answered"] == False:
        texts = data.pop("text")

        for chapter_idx, text in enumerate(tqdm(texts)):
            answer_list = []
            for it in data[str(chapter_idx)]:
                # print("question:", it)
                answer = generate_answer(question_text=it, information_text=text)
                # print("answer:", answer)
                answer_list.append(answer)

            data[f"{chapter_idx}_answers"] = answer_list

        data["answered"] = True
        data["distilled"] = False
        data["text"] = texts
        with open(path, "w") as json_file:
            json.dump(data, json_file, indent=4)


def write_answer_to_dataset(path):
    with open(path, "r") as json_file:
        data = json.load(json_file)
    if data["type"] == "journal" and data["answered"] == False:
        __write_answer_to_journal_dataset(path)
    elif data["type"] == "book" and data["answered"] == False:
        __write_answer_to_book_dataset(path)




In [5]:
write_answer_to_dataset("./outputs/Small Methods - 2024 - Diao - AI‐Equipped Scanning Probe Microscopy for Autonomous Site‐Specific Atomic‐Level.pdf.json")

  0%|          | 0/39 [00:00<?, ?it/s]The attention mask is not set and cannot be inferred from input because pad token is same as eos token. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.


question: What is the primary purpose of integrating AI into scanning probe microscopy (SPM) systems?
input token count is 10666


  3%|▎         | 1/39 [00:27<17:27, 27.57s/it]

answer: The primary purpose of integrating artificial intelligence (AI) into scanning probe microscopy (SPM) systems is to enhance the precision, efficiency, and reliability of atomic-scale measurements and surface analysis. AI-equipped SPM systems, such as the AI-SPM described in the article, are designed to autonomously identify and manipulate atomic positions with high precision. This integration allows for several key advancements:

1. **Autonomous Operation**: AI enables the SPM system to perform complex tasks such as spectroscopic data acquisition and atomic adjustment without constant human intervention. This includes the ability to detect and adapt to surface defects, targeting or avoiding them as necessary.

2. **Overcoming Challenges**: AI helps overcome typical challenges associated with SPM, such as positional drift and tip apex atomic variations due to thermal effects. This ensures accurate, site-specific surface analysis even under demanding conditions like room temperatu

  5%|▌         | 2/39 [00:56<17:32, 28.45s/it]

...

answer: In the AI-Enhanced Scanning Probe Microscopy (AI-SPM) system, the cosine similarity metric plays a crucial role in the statistical analysis of current-voltage (\(I\)-\(V\)) spectroscopy data. Here's a detailed explanation of its role:

1. **Purpose of Cosine Similarity**: The cosine similarity metric is used to measure the similarity between two \(I\)-\(V\) curves. It provides a quantitative way to assess how closely two curves resemble each other, with a value ranging from -1 to 1. A value of 1 indicates that the curves are identical, while a value of -1 indicates that they are completely dissimilar.

2. **Data Analysis Context**: In the context of the AI-SPM system, the \(I\)-\(V\) curves are obtained from measurements at different atomic sites on a surface. Due to the inherent variability in room temperature measurements, such as thermal drift and tip apex changes, the data can exhibit significant noise and variability.

3. **Extracting Consistent Trends**: The cosine simila

 97%|█████████▋| 38/39 [21:03<00:28, 28.47s/it]

answer: The AI-Enhanced Scanning Probe Microscopy (AI-SPM) system ensures the accuracy and reliability of its measurements through several advanced techniques and strategies:

1. **Deep Learning Models**: The system employs convolutional neural networks (CNNs) tailored for specific tasks, such as recognizing tip and sample conditions, identifying adsorbates, and detecting site-specific information. These models are trained on extensive datasets to achieve high accuracy, with reported accuracies of 87-90% for tip and sample condition recognition, 92% for adsorbate identification, and 91% for atomic site detection.

2. **Two-Phase Training Data Acquisition**: To handle the variability and instability of room temperature measurements, the AI-SPM system uses a two-phase approach to data acquisition. The first phase involves collecting diverse datasets under various conditions, including non-ideal images, to train the models comprehensively. The second phase expands the dataset with high-qu

100%|██████████| 39/39 [21:25<00:00, 32.96s/it]

answer: Integrating artificial intelligence (AI) into scanning probe microscopy (SPM) technologies offers several key benefits that enhance the capabilities and efficiency of material characterization at the atomic level. Here are some of the primary advantages:

1. **Autonomous Operation**: AI enables SPM systems to perform autonomous measurements, reducing the need for constant human intervention. This is particularly beneficial for time-consuming and repetitive tasks, allowing researchers to focus on data analysis and interpretation.

2. **Enhanced Precision and Accuracy**: AI algorithms can process and analyze data in real-time, allowing for more precise and accurate measurements. This is crucial for identifying and manipulating atomic positions, especially in environments where thermal drift and tip apex variations are prevalent.

3. **Adaptability to Environmental Conditions**: AI-equipped SPM systems can adapt to varying environmental conditions, such as room temperature, where 